# Walk-Forward Long-Cash Mean Reversion

Extends the long/cash mean-reversion rule with walk-forward parameter selection.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
from _market_data import download_price_records
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")


class StrategyConfig:
    """
    Configuration class for Larry Connors MDU/MDD Strategy
    All parameters are exposed and configurable by the user

    ENHANCEMENTS:
    - Added hyperparameter grid for walk-forward optimization
    - Added DSR/PSR calculation parameters
    - Added universe fetching from Yahoo Finance
    """

    MARKET_CAP_MIN = 1000000000.0
    TICKERS = ["SPY", "XLK"]
    BENCHMARK_TICKER = "URTH"
    END_DATE = "2024-12-31"
    START_DATE = "2019-01-01"
    MA_LONG_PERIOD = 200
    MA_SHORT_PERIOD = 5
    LOOKBACK_DAYS = 5
    MIN_DOWN_DAYS = 4
    PARAM_GRID = {
        "MA_SHORT_PERIOD": [3, 5, 7],
        "MIN_DOWN_DAYS": [3, 4, 5],
        "MA_LONG_PERIOD": [150, 200, 250],
    }
    INITIAL_CAPITAL = 100000
    COMPOUNDING_MODE = False
    LEVERAGE = 1.0
    COMMISSION_PCT = 0.001
    TRAIN_YEARS = 2
    VALIDATION_YEARS = 1
    TEST_YEARS = 1
    REOPTIMIZE_FREQ = "yearly"
    RISK_FREE_RATE = 0.02
    TARGET_CONFIDENCE = 0.95
    NUM_STRATEGY_TESTS = 10
    BACKTEST_VARIANCE = 0.0
    FIGURE_SIZE = (14, 8)
    SAVE_FIGURES = False
    FIGURE_DPI = 300


def download_fmp_data(ticker, start_date, end_date):
    """
    Download historical OHLCV data from Yahoo Finance.

    CRITICAL: No synthetic data or fallback - returns None if download fails.
    This ensures we only use real market data for backtesting.
    """
    try:
        response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
        data = response_prices
        if "historical" not in data or len(data["historical"]) == 0:
            print(f"WARNING: No data returned for {ticker}")
            return None
        df = pd.DataFrame(data["historical"])
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").reset_index(drop=True)
        df = df[["date", "open", "high", "low", "close", "volume"]]
        return df
    except Exception as e:
        print(f"ERROR downloading {ticker}: {str(e)}")
        return None


def load_all_data(config):
    """
    Load data for all tickers and benchmark.

    ENHANCEMENT: Now supports Yahoo Finance universe fetching or static list.
    """
    print("\n" + "=" * 80)
    print("DATA DOWNLOAD MODULE")
    print("=" * 80)
    all_data = {}
    universe = config.TICKERS
    valid_tickers = [t for t in universe if t and len(t) > 0]
    if config.BENCHMARK_TICKER not in valid_tickers:
        valid_tickers.append(config.BENCHMARK_TICKER)
    print(f"→ Downloading data for {len(valid_tickers)} tickers...")
    for ticker in valid_tickers:
        df = download_fmp_data(ticker, config.START_DATE, config.END_DATE)
        if df is not None and len(df) > 0:
            all_data[ticker] = df
    print("=" * 80)
    print(f"Successfully loaded {len(all_data)} instruments\n")
    return all_data


def calculate_features(df, ma_long=200, ma_short=5, lookback_days=5):
    """
    Calculate technical indicators for strategy signals.

    CRITICAL: All features use T-1 data to avoid look-ahead bias.
    - Moving averages are calculated on close prices
    - Down days counter looks at prior N days only
    - No future data is used in any calculation

    ENHANCEMENT: Now accepts parameters for hyperparameter optimization
    """
    df = df.copy()
    df["returns"] = df["close"].pct_change()
    df["ma_long"] = df["close"].rolling(window=ma_long, min_periods=ma_long).mean()
    df["ma_short"] = df["close"].rolling(window=ma_short, min_periods=ma_short).mean()
    df["is_down_day"] = (df["close"] < df["close"].shift(1)).astype(int)
    df["down_days_count"] = (
        df["is_down_day"].rolling(window=lookback_days, min_periods=lookback_days).sum()
    )
    df["ma_long_lag"] = df["ma_long"].shift(1)
    df["ma_short_lag"] = df["ma_short"].shift(1)
    df["close_lag"] = df["close"].shift(1)
    df["down_days_count_lag"] = df["down_days_count"].shift(1)
    return df


def generate_signals(df, min_down_days=4):
    """
    Generate buy/sell signals based on Connors MDU/MDD rules.

    FIXED: Use .iloc instead of .loc for integer-based indexing
    to avoid KeyError when index is not sequential.

    ENTRY RULES (all must be true at T-1 close):
    1. Close > long-term MA (long-term uptrend confirmation)
    2. Close < short-term MA (short-term pullback)
    3. At least min_down_days out of last lookback_days were down days

    EXIT RULES:
    - Exit when Close > short-term MA (mean reversion complete)

    NO STOP-LOSS per original Connors strategy

    TIMING:
    - Signal generated at T-1 close
    - Order executed at T open (next day)
    - This prevents look-ahead bias

    ENHANCEMENT: Now accepts parameters for hyperparameter optimization
    """
    df = df.copy()
    df = df.reset_index(drop=True)
    df["signal"] = 0
    condition_trend = df["close_lag"] > df["ma_long_lag"]
    condition_pullback = df["close_lag"] < df["ma_short_lag"]
    condition_down_days = df["down_days_count_lag"] >= min_down_days
    entry_signal = condition_trend & condition_pullback & condition_down_days
    exit_signal = df["close_lag"] >= df["ma_short_lag"]
    position = 0
    for i in range(len(df)):
        if pd.isna(df.iloc[i]["ma_long_lag"]) or pd.isna(df.iloc[i]["ma_short_lag"]):
            df.loc[i, "signal"] = 0
            position = 0
            continue
        if position == 1 and exit_signal.iloc[i]:
            position = 0
        elif position == 0 and entry_signal.iloc[i]:
            position = 1
        df.loc[i, "signal"] = position
    df["position"] = df["signal"].shift(1).fillna(0)
    return df


def calculate_position_size(portfolio_value, current_capital, compounding_mode, leverage):
    """
    Calculate position size based on sizing methodology.

    CRITICAL: Position sizing methodology to avoid artificial leverage bug

    TWO MODES:
    1. COMPOUNDING_MODE = False (RESEARCH MODE - DEFAULT):
       - Size positions on INITIAL_CAPITAL (fixed notional)
       - Maintains constant dollar exposure throughout backtest
       - Returns reflect pure signal alpha without leverage drift
       - Use for: Alpha research, strategy comparison, market-neutral

    2. COMPOUNDING_MODE = True (PRODUCTION MODE):
       - Size positions on current portfolio value
       - Simulates real fund with reinvestment
       - Returns are geometrically compounded
       - Use for: Investor presentations, AUM projections

    LEVERAGE parameter controls risk level independently of compounding
    """
    if compounding_mode:
        base_size = portfolio_value
    else:
        base_size = current_capital
    position_size = base_size * leverage
    return position_size


def backtest_strategy(df, initial_capital, compounding_mode, leverage, commission_pct, ticker):
    """
    Execute backtest with realistic execution and cost assumptions.

    EXECUTION TIMING (strict T-1 to T logic):
    - Day T-1 Close: Signals generated using data up to T-1
    - Day T Open: Orders executed at open price
    - Day T Close: Portfolio marked-to-market at close price

    COST MODEL:
    - Entry commission: commission_pct * position_value
    - Exit commission: commission_pct * position_value
    - No slippage assumed (conservative for liquid ETFs)

    ENHANCEMENT: Fixed cash handling per requirements
    - Investable amount considers commission upfront
    - Shares are rounded down to keep cash non-negative
    - No implicit borrowing occurs

    FIXED: Use .iloc for positional indexing and ensure reset_index.

    NO STOP-LOSS per Connors original strategy
    """
    df = df.copy()
    df = df.reset_index(drop=True)
    portfolio_value = initial_capital
    cash = initial_capital
    shares = 0
    entry_price = 0
    entry_commission = 0
    df["portfolio_value"] = initial_capital
    df["cash"] = initial_capital
    df["shares"] = 0
    df["trade"] = ""
    df["pnl"] = 0
    df["commission"] = 0
    for i in range(1, len(df)):
        current_position = df.iloc[i]["position"]
        previous_position = df.iloc[i - 1]["position"]
        open_price = df.iloc[i]["open"]
        close_price = df.iloc[i]["close"]
        if current_position == 1 and previous_position == 0:
            position_size = calculate_position_size(
                portfolio_value=df.iloc[i - 1]["portfolio_value"],
                current_capital=initial_capital,
                compounding_mode=compounding_mode,
                leverage=leverage,
            )
            investable = position_size / (1 + commission_pct)
            shares = int(investable / open_price)
            if shares > 0:
                actual_position_value = shares * open_price
                entry_price = open_price
                entry_commission = actual_position_value * commission_pct
                cash = cash - actual_position_value - entry_commission
                df.loc[i, "trade"] = "BUY"
                df.loc[i, "commission"] = entry_commission
        elif current_position == 0 and previous_position == 1 and (shares > 0):
            exit_value = shares * open_price
            exit_commission = exit_value * commission_pct
            pnl = exit_value - shares * entry_price - entry_commission - exit_commission
            cash = cash + exit_value - exit_commission
            df.loc[i, "trade"] = "SELL"
            df.loc[i, "pnl"] = pnl
            df.loc[i, "commission"] = exit_commission
            shares = 0
            entry_price = 0
            entry_commission = 0
        if shares > 0:
            portfolio_value = cash + shares * close_price
        else:
            portfolio_value = cash
        df.loc[i, "portfolio_value"] = portfolio_value
        df.loc[i, "cash"] = cash
        df.loc[i, "shares"] = shares
    df["strategy_returns"] = df["portfolio_value"].pct_change().fillna(0)
    df["bh_returns"] = df["close"].pct_change().fillna(0)
    df["strategy_cumulative"] = (1 + df["strategy_returns"]).cumprod()
    df["bh_cumulative"] = (1 + df["bh_returns"]).cumprod()
    df["cumulative_max"] = df["portfolio_value"].expanding().max()
    df["drawdown"] = (df["portfolio_value"] - df["cumulative_max"]) / df["cumulative_max"]
    return df


def calculate_deflated_sharpe_ratio(
    sharpe_ratio, n_obs, n_tests, skewness, kurtosis, backtest_variance=0.0
):
    """
    ADDED: Calculate Bailey et al. Deflated Sharpe Ratio (DSR).

    The DSR adjusts the observed Sharpe ratio for:
    1. Multiple testing (trying many strategies)
    2. Non-normality of returns (skew and kurtosis)
    3. Estimation uncertainty

    This is a more rigorous implementation than dsr = SR - sqrt(2*log(M)/n)

    Reference: Bailey, D. H., & López de Prado, M. (2014).
    "The Deflated Sharpe Ratio: Correcting for Selection Bias, Backtest Overfitting, and Non-Normality"
    """
    if n_obs < 3 or n_tests < 1:
        return (0.0, 0.0, 0.0)
    var_sr = (
        1 + 0.5 * sharpe_ratio**2 - skewness * sharpe_ratio + (kurtosis - 3) / 4 * sharpe_ratio**2
    ) / (n_obs - 1)
    var_sr += backtest_variance
    if var_sr <= 0:
        return (0.0, 0.0, 0.0)
    threshold_sr = np.sqrt(var_sr) * (
        (1 - np.euler_gamma) * stats.norm.ppf(1 - 1 / n_tests)
        + np.euler_gamma * stats.norm.ppf(1 - 1 / (n_tests * np.e))
    )
    dsr = (sharpe_ratio - threshold_sr) / np.sqrt(var_sr)
    prob_skill = stats.norm.cdf(dsr)
    return (dsr, threshold_sr, prob_skill)


def calculate_probabilistic_sharpe_ratio(sharpe_ratio, n_obs, skewness, kurtosis, benchmark_sr=0.0):
    """
    Calculate Probabilistic Sharpe Ratio (PSR).

    PSR measures the probability that the true Sharpe ratio exceeds a benchmark.
    Adjusts for non-normality via skewness and kurtosis.
    """
    if n_obs < 3:
        return 0.0
    var_sr = (
        1 + 0.5 * sharpe_ratio**2 - skewness * sharpe_ratio + (kurtosis - 3) / 4 * sharpe_ratio**2
    ) / (n_obs - 1)
    if var_sr <= 0:
        return 0.0
    z_score = (sharpe_ratio - benchmark_sr) / np.sqrt(var_sr)
    psr = stats.norm.cdf(z_score)
    return psr


def calculate_performance_metrics(df, initial_capital, risk_free_rate, ticker, n_tests=10):
    """
    Calculate comprehensive performance statistics.

    ENHANCEMENT: Fixed metrics rebasing per requirements
    - Now uses first portfolio value of the period as base
    - Total/annualized returns calculated correctly per block
    - DSR calculation improved per Bailey et al.
    """
    if len(df) == 0:
        return {
            "ticker": ticker,
            "total_return_pct": 0,
            "annualized_return_pct": 0,
            "volatility_pct": 0,
            "sharpe_ratio": 0,
            "sortino_ratio": 0,
            "max_drawdown_pct": 0,
            "calmar_ratio": 0,
            "win_rate_pct": 0,
            "profit_factor": 0,
            "avg_trade_pnl": 0,
            "num_trades": 0,
            "psr": 0,
            "dsr": 0,
            "dsr_threshold": 0,
            "prob_skill": 0,
            "final_portfolio_value": initial_capital,
        }
    start_pv = df["portfolio_value"].iloc[0]
    final_pv = df["portfolio_value"].iloc[-1]
    total_return = (final_pv / start_pv - 1) * 100
    days = len(df)
    years = days / 252
    if years > 0 and start_pv > 0:
        annualized_return = ((final_pv / start_pv) ** (1 / years) - 1) * 100
    else:
        annualized_return = 0
    volatility = df["strategy_returns"].std() * np.sqrt(252) * 100 if len(df) > 1 else 0
    excess_returns = df["strategy_returns"] - risk_free_rate / 252
    sharpe_ratio = (
        excess_returns.mean() / df["strategy_returns"].std() * np.sqrt(252)
        if df["strategy_returns"].std() > 0
        else 0
    )
    negative_returns = df["strategy_returns"][df["strategy_returns"] < 0]
    downside_std = (
        negative_returns.std() if len(negative_returns) > 0 else df["strategy_returns"].std()
    )
    sortino_ratio = excess_returns.mean() / downside_std * np.sqrt(252) if downside_std > 0 else 0
    cumulative = (1 + df["strategy_returns"]).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min() * 100
    calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0
    trades = df[df["trade"].isin(["BUY", "SELL"])]
    num_trades = len(trades[trades["trade"] == "BUY"])
    closed_trades = df[df["trade"] == "SELL"]
    if len(closed_trades) > 0:
        winning_trades = closed_trades[closed_trades["pnl"] > 0]
        losing_trades = closed_trades[closed_trades["pnl"] < 0]
        win_rate = len(winning_trades) / len(closed_trades) * 100
        avg_trade_pnl = closed_trades["pnl"].mean()
        total_wins = winning_trades["pnl"].sum() if len(winning_trades) > 0 else 0
        total_losses = abs(losing_trades["pnl"].sum()) if len(losing_trades) > 0 else 1
        profit_factor = total_wins / total_losses if total_losses > 0 else 0
    else:
        win_rate = 0
        avg_trade_pnl = 0
        profit_factor = 0
    n = len(df)
    skewness = df["strategy_returns"].skew() if n > 2 else 0
    kurtosis = df["strategy_returns"].kurtosis() if n > 3 else 0
    psr = calculate_probabilistic_sharpe_ratio(
        sharpe_ratio=sharpe_ratio, n_obs=n, skewness=skewness, kurtosis=kurtosis, benchmark_sr=0.0
    )
    dsr, dsr_threshold, prob_skill = calculate_deflated_sharpe_ratio(
        sharpe_ratio=sharpe_ratio,
        n_obs=n,
        n_tests=n_tests,
        skewness=skewness,
        kurtosis=kurtosis,
        backtest_variance=0.0,
    )
    metrics = {
        "ticker": ticker,
        "total_return_pct": total_return,
        "annualized_return_pct": annualized_return,
        "volatility_pct": volatility,
        "sharpe_ratio": sharpe_ratio,
        "sortino_ratio": sortino_ratio,
        "max_drawdown_pct": max_drawdown,
        "calmar_ratio": calmar_ratio,
        "win_rate_pct": win_rate,
        "profit_factor": profit_factor,
        "avg_trade_pnl": avg_trade_pnl,
        "num_trades": num_trades,
        "psr": psr,
        "dsr": dsr,
        "dsr_threshold": dsr_threshold,
        "prob_skill": prob_skill,
        "final_portfolio_value": final_pv,
        "skewness": skewness,
        "kurtosis": kurtosis,
    }
    return metrics


def calculate_benchmark_metrics(df, initial_capital, risk_free_rate, ticker):
    """
    Calculate buy-and-hold benchmark performance.

    ENHANCEMENT: Fixed rebasing for subperiod calculations
    """
    if len(df) == 0:
        return {
            "ticker": ticker,
            "total_return_pct": 0,
            "annualized_return_pct": 0,
            "volatility_pct": 0,
            "sharpe_ratio": 0,
            "max_drawdown_pct": 0,
        }
    df = df.copy()
    df["returns"] = df["close"].pct_change().fillna(0)
    start_price = df["close"].iloc[0]
    end_price = df["close"].iloc[-1]
    total_return = (end_price / start_price - 1) * 100
    days = len(df)
    years = days / 252
    annualized_return = ((end_price / start_price) ** (1 / years) - 1) * 100 if years > 0 else 0
    volatility = df["returns"].std() * np.sqrt(252) * 100
    excess_returns = df["returns"] - risk_free_rate / 252
    sharpe_ratio = (
        excess_returns.mean() / df["returns"].std() * np.sqrt(252) if df["returns"].std() > 0 else 0
    )
    cumulative = (1 + df["returns"]).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min() * 100
    metrics = {
        "ticker": ticker,
        "total_return_pct": total_return,
        "annualized_return_pct": annualized_return,
        "volatility_pct": volatility,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown_pct": max_drawdown,
    }
    return metrics


def optimize_hyperparameters(train_df, val_df, config, ticker):
    """
    ADDED: Perform hyperparameter optimization using train data and select best on validation.

    This implements a compact grid search over specified hyperparameters:
    - MA_SHORT_PERIOD
    - MIN_DOWN_DAYS
    - MA_LONG_PERIOD

    Selection criteria (in priority order):
    1. Sharpe ratio on validation set (primary)
    2. If tie, prefer lower drawdown
    3. If still tie, prefer lower turnover

    This prevents overfitting to training data by requiring validation confirmation.
    """
    print(f"  → Optimizing hyperparameters for {ticker}...")
    grid = config.PARAM_GRID
    param_combinations = list(
        product(grid["MA_SHORT_PERIOD"], grid["MIN_DOWN_DAYS"], grid["MA_LONG_PERIOD"])
    )
    best_params = None
    best_val_sharpe = -np.inf
    best_val_drawdown = -np.inf
    results = []
    for params in param_combinations:
        ma_short, min_down, ma_long = params
        if ma_short >= ma_long:
            continue
        try:
            train_featured = calculate_features(
                train_df.copy(),
                ma_long=ma_long,
                ma_short=ma_short,
                lookback_days=config.LOOKBACK_DAYS,
            )
            train_signals = generate_signals(train_featured, min_down_days=min_down)
            train_backtest = backtest_strategy(
                train_signals,
                initial_capital=config.INITIAL_CAPITAL,
                compounding_mode=config.COMPOUNDING_MODE,
                leverage=config.LEVERAGE,
                commission_pct=config.COMMISSION_PCT,
                ticker=ticker,
            )
            train_metrics = calculate_performance_metrics(
                train_backtest,
                initial_capital=config.INITIAL_CAPITAL,
                risk_free_rate=config.RISK_FREE_RATE,
                ticker=ticker,
                n_tests=len(param_combinations),
            )
            val_featured = calculate_features(
                val_df.copy(),
                ma_long=ma_long,
                ma_short=ma_short,
                lookback_days=config.LOOKBACK_DAYS,
            )
            val_signals = generate_signals(val_featured, min_down_days=min_down)
            val_start_capital = train_backtest["portfolio_value"].iloc[-1]
            val_backtest = backtest_strategy(
                val_signals,
                initial_capital=val_start_capital,
                compounding_mode=config.COMPOUNDING_MODE,
                leverage=config.LEVERAGE,
                commission_pct=config.COMMISSION_PCT,
                ticker=ticker,
            )
            val_metrics = calculate_performance_metrics(
                val_backtest,
                initial_capital=val_start_capital,
                risk_free_rate=config.RISK_FREE_RATE,
                ticker=ticker,
                n_tests=len(param_combinations),
            )
            results.append(
                {
                    "ma_short": ma_short,
                    "min_down": min_down,
                    "ma_long": ma_long,
                    "train_sharpe": train_metrics["sharpe_ratio"],
                    "val_sharpe": val_metrics["sharpe_ratio"],
                    "val_return": val_metrics["annualized_return_pct"],
                    "val_drawdown": val_metrics["max_drawdown_pct"],
                    "val_trades": val_metrics["num_trades"],
                }
            )
            if val_metrics["sharpe_ratio"] > best_val_sharpe or (
                val_metrics["sharpe_ratio"] == best_val_sharpe
                and val_metrics["max_drawdown_pct"] > best_val_drawdown
            ):
                best_val_sharpe = val_metrics["sharpe_ratio"]
                best_val_drawdown = val_metrics["max_drawdown_pct"]
                best_params = {
                    "MA_SHORT_PERIOD": ma_short,
                    "MIN_DOWN_DAYS": min_down,
                    "MA_LONG_PERIOD": ma_long,
                    "train_sharpe": train_metrics["sharpe_ratio"],
                    "val_sharpe": val_metrics["sharpe_ratio"],
                    "val_return": val_metrics["annualized_return_pct"],
                    "val_drawdown": val_metrics["max_drawdown_pct"],
                    "val_trades": val_metrics["num_trades"],
                }
        except Exception as e:
            continue
    if best_params is None:
        print(f"    WARNING: No valid parameters found, using defaults")
        best_params = {
            "MA_SHORT_PERIOD": config.MA_SHORT_PERIOD,
            "MIN_DOWN_DAYS": config.MIN_DOWN_DAYS,
            "MA_LONG_PERIOD": config.MA_LONG_PERIOD,
            "train_sharpe": 0,
            "val_sharpe": 0,
            "val_return": 0,
            "val_drawdown": 0,
            "val_trades": 0,
        }
    print(
        f"    ✓ Best params: MA_SHORT={best_params['MA_SHORT_PERIOD']}, "
        + f"MIN_DOWN={best_params['MIN_DOWN_DAYS']}, MA_LONG={best_params['MA_LONG_PERIOD']}"
    )
    print(
        f"    ✓ Val Sharpe: {best_params['val_sharpe']:.2f}, Val Return: {best_params['val_return']:.1f}%"
    )
    return best_params


def walk_forward_analysis(df, config, ticker):
    """
    ENHANCED: Perform walk-forward analysis with train → validate → test structure.

    FIXED: Reset index after filtering by year to ensure sequential integer indexing.
    This prevents KeyError when using .loc[i, ...] operations.

    NEW METHODOLOGY:
    - Expanding window approach (uses all prior data)
    - Each window split into: train (e.g., 2 years), validate (e.g., 1 year), test (next year)
    - Hyperparameters optimized on train, selected on validate
    - Final evaluation on test (completely unseen)
    - Re-optimization each block to prevent temporal overfitting

    STRUCTURE:
    - Train period: config.TRAIN_YEARS years
    - Validation period: config.VALIDATION_YEARS years
    - Test period: config.TEST_YEARS years
    - Repeat with expanding train window

    METRICS STORED PER BLOCK:
    - params_used: Locked parameters for test
    - sharpe_in_sample: Train Sharpe
    - sharpe_validation: Validation Sharpe
    - sharpe_out_of_sample: Test Sharpe (OOS)
    - psr_out_of_sample: PSR on test
    - dsr_out_of_sample: DSR on test
    - Other standard metrics on test period
    """
    print(f"\n→ Running walk-forward analysis for {ticker}...")
    df["year"] = df["date"].dt.year
    years = sorted(df["year"].unique())
    if len(years) < config.TRAIN_YEARS + config.VALIDATION_YEARS + config.TEST_YEARS:
        print(f"  WARNING: Insufficient data for walk-forward analysis on {ticker}")
        print(
            f"  Need at least {config.TRAIN_YEARS + config.VALIDATION_YEARS + config.TEST_YEARS} years, have {len(years)}"
        )
        return []
    results = []
    train_end_year = years[0] + config.TRAIN_YEARS - 1
    val_end_year = train_end_year + config.VALIDATION_YEARS
    while val_end_year < years[-1]:
        test_end_year = val_end_year + config.TEST_YEARS
        if test_end_year > years[-1]:
            break
        train_data = df[df["year"] <= train_end_year].copy().reset_index(drop=True)
        val_data = (
            df[(df["year"] > train_end_year) & (df["year"] <= val_end_year)]
            .copy()
            .reset_index(drop=True)
        )
        test_data = (
            df[(df["year"] > val_end_year) & (df["year"] <= test_end_year)]
            .copy()
            .reset_index(drop=True)
        )
        if len(train_data) < 252 or len(val_data) < 60 or len(test_data) < 60:
            val_end_year += config.TEST_YEARS
            continue
        period_label = f"Train:{years[0]}-{train_end_year}, Val:{train_end_year + 1}-{val_end_year}, Test:{val_end_year + 1}-{test_end_year}"
        print(f"\n  Block: {period_label}")
        best_params = optimize_hyperparameters(train_data, val_data, config, ticker)
        test_featured = calculate_features(
            test_data.copy(),
            ma_long=best_params["MA_LONG_PERIOD"],
            ma_short=best_params["MA_SHORT_PERIOD"],
            lookback_days=config.LOOKBACK_DAYS,
        )
        test_signals = generate_signals(test_featured, min_down_days=best_params["MIN_DOWN_DAYS"])
        test_start_capital = config.INITIAL_CAPITAL
        test_backtest = backtest_strategy(
            test_signals,
            initial_capital=test_start_capital,
            compounding_mode=config.COMPOUNDING_MODE,
            leverage=config.LEVERAGE,
            commission_pct=config.COMMISSION_PCT,
            ticker=ticker,
        )
        test_metrics = calculate_performance_metrics(
            test_backtest,
            initial_capital=test_start_capital,
            risk_free_rate=config.RISK_FREE_RATE,
            ticker=ticker,
            n_tests=config.NUM_STRATEGY_TESTS,
        )
        block_result = {
            "ticker": ticker,
            "period": period_label,
            "test_years": f"{val_end_year + 1}-{test_end_year}",
            "train_size": len(train_data),
            "val_size": len(val_data),
            "test_size": len(test_data),
            "params_ma_short": best_params["MA_SHORT_PERIOD"],
            "params_min_down": best_params["MIN_DOWN_DAYS"],
            "params_ma_long": best_params["MA_LONG_PERIOD"],
            "sharpe_in_sample": best_params["train_sharpe"],
            "sharpe_validation": best_params["val_sharpe"],
            "return_validation": best_params["val_return"],
            "sharpe_out_of_sample": test_metrics["sharpe_ratio"],
            "return_out_of_sample": test_metrics["annualized_return_pct"],
            "volatility_out_of_sample": test_metrics["volatility_pct"],
            "drawdown_out_of_sample": test_metrics["max_drawdown_pct"],
            "calmar_out_of_sample": test_metrics["calmar_ratio"],
            "win_rate_out_of_sample": test_metrics["win_rate_pct"],
            "num_trades_out_of_sample": test_metrics["num_trades"],
            "psr_out_of_sample": test_metrics["psr"],
            "dsr_out_of_sample": test_metrics["dsr"],
            "dsr_threshold_out_of_sample": test_metrics["dsr_threshold"],
            "prob_skill_out_of_sample": test_metrics["prob_skill"],
        }
        results.append(block_result)
        print(
            f"    → Test OOS: Return={test_metrics['annualized_return_pct']:.1f}%, "
            + f"Sharpe={test_metrics['sharpe_ratio']:.2f}, Trades={test_metrics['num_trades']}"
        )
        print(f"    → Overfitting: PSR={test_metrics['psr']:.1%}, DSR={test_metrics['dsr']:.2f}")
        val_end_year += config.TEST_YEARS
    print(f"  ✓ Completed {len(results)} walk-forward blocks for {ticker}")
    return results


def run_strategy():
    """
    Main execution function for Larry Connors MDU/MDD strategy.

    ENHANCED WORKFLOW:
    1. Load configuration
    2. Fetch universe from Yahoo Finance or use static list (NEW)
    3. Download data from Yahoo Finance
    4. Run walk-forward analysis with train→validate→test (ENHANCED + FIXED)
    5. Hyperparameter optimization per block (NEW)
    6. Calculate performance with fixed rebasing (FIXED)
    7. Compute enhanced DSR/PSR per Bailey et al. (ENHANCED)
    8. Compare with URTH benchmark (FIXED)
    9. Print comprehensive report
    """
    config = StrategyConfig()
    print("\n" + "=" * 80)
    print("LARRY CONNORS MDU/MDD MEAN REVERSION STRATEGY")
    print("Enhanced Walk-Forward Backtesting Framework")
    print("=" * 80)
    print("\nFEATURES:")
    print("✓ Train → Validate → Test walk-forward analysis")
    print("✓ Hyperparameter optimization per block")
    print("✓ Bailey et al. Deflated Sharpe Ratio (DSR)")
    print("✓ Probabilistic Sharpe Ratio (PSR)")
    print("✓ Fixed position sizing (no artificial leverage)")
    print("✓ Correct metrics rebasing per period")
    print(f"✓ Benchmark: {config.BENCHMARK_TICKER}")
    print("=" * 80)
    all_data = load_all_data(config)
    if len(all_data) == 0:
        print("\nERROR: No data available. Cannot proceed with backtest.")
        return
    all_wf_results = []
    valid_tickers = [t for t in config.TICKERS if t and len(t) > 0]
    for ticker in valid_tickers:
        if ticker not in all_data:
            print(f"\nSkipping {ticker} - no data available")
            continue
        print(f"\n{'=' * 80}")
        print(f"PROCESSING: {ticker}")
        print(f"{'=' * 80}")
        df = all_data[ticker].copy()
        wf_results = walk_forward_analysis(df, config, ticker)
        all_wf_results.extend(wf_results)
    benchmark_results = []
    if config.BENCHMARK_TICKER in all_data:
        print(f"\n{'=' * 80}")
        print(f"PROCESSING BENCHMARK: {config.BENCHMARK_TICKER}")
        print(f"{'=' * 80}")
        benchmark_df = all_data[config.BENCHMARK_TICKER].copy()
        for wf_result in all_wf_results:
            test_years = wf_result["test_years"]
            year_range = test_years.split("-")
            if len(year_range) == 2:
                start_year = int(year_range[0])
                end_year = int(year_range[1])
                bench_period = benchmark_df[
                    (benchmark_df["date"].dt.year >= start_year)
                    & (benchmark_df["date"].dt.year <= end_year)
                ].copy()
                if len(bench_period) > 0:
                    bench_metrics = calculate_benchmark_metrics(
                        bench_period,
                        initial_capital=config.INITIAL_CAPITAL,
                        risk_free_rate=config.RISK_FREE_RATE,
                        ticker=config.BENCHMARK_TICKER,
                    )
                    bench_metrics["period"] = wf_result["period"]
                    benchmark_results.append(bench_metrics)
    print("\n" + "=" * 80)
    print("WALK-FORWARD ANALYSIS RESULTS")
    print("=" * 80)
    if len(all_wf_results) > 0:
        wf_df = pd.DataFrame(all_wf_results)
        print("\nOVERALL STATISTICS (Across All OOS Periods):")
        print("-" * 80)
        print(f"Number of blocks:          {len(wf_df)}")
        print(f"Avg OOS Return:            {wf_df['return_out_of_sample'].mean():.2f}%")
        print(f"Std OOS Return:            {wf_df['return_out_of_sample'].std():.2f}%")
        print(f"Avg OOS Sharpe:            {wf_df['sharpe_out_of_sample'].mean():.2f}")
        print(f"Avg OOS Max DD:            {wf_df['drawdown_out_of_sample'].mean():.2f}%")
        print(
            f"Positive blocks:           {(wf_df['return_out_of_sample'] > 0).sum()}/{len(wf_df)}"
        )
        print(f"\nOVERFITTING METRICS:")
        print(f"Avg PSR (OOS):             {wf_df['psr_out_of_sample'].mean():.1%}")
        print(f"Avg DSR (OOS):             {wf_df['dsr_out_of_sample'].mean():.2f}")
        print(f"Avg Prob of Skill:         {wf_df['prob_skill_out_of_sample'].mean():.1%}")
        print("\nBLOCK-BY-BLOCK RESULTS:")
        print("-" * 80)
        display_cols = [
            "period",
            "params_ma_short",
            "params_min_down",
            "params_ma_long",
            "sharpe_in_sample",
            "sharpe_validation",
            "sharpe_out_of_sample",
            "return_out_of_sample",
            "psr_out_of_sample",
            "dsr_out_of_sample",
        ]
        pd.set_option("display.max_columns", None)
        pd.set_option("display.width", None)
        pd.set_option("display.max_colwidth", 50)
        print(wf_df[display_cols].to_string(index=False))
        if len(benchmark_results) > 0:
            print(f"\nBENCHMARK COMPARISON ({config.BENCHMARK_TICKER}):")
            print("-" * 80)
            bench_df = pd.DataFrame(benchmark_results)
            for idx, row in wf_df.iterrows():
                matching_bench = bench_df[bench_df["period"] == row["period"]]
                if len(matching_bench) > 0:
                    bench_return = matching_bench.iloc[0]["annualized_return_pct"]
                    bench_sharpe = matching_bench.iloc[0]["sharpe_ratio"]
                    strat_return = row["return_out_of_sample"]
                    strat_sharpe = row["sharpe_out_of_sample"]
                    print(f"\n{row['test_years']}:")
                    print(f"  Strategy: Return={strat_return:.1f}%, Sharpe={strat_sharpe:.2f}")
                    print(
                        f"  {config.BENCHMARK_TICKER}: Return={bench_return:.1f}%, Sharpe={bench_sharpe:.2f}"
                    )
                    print(f"  Excess Return: {strat_return - bench_return:.1f}%")
    print("\n" + "=" * 80)
    print("ANALYSIS COMPLETE")
    print("=" * 80)


if __name__ == "__main__":
    run_strategy()
